In [ ]:
import requests

# 1. 캡처 화면에 있던 '인증키'를 복사해서 아래 따옴표 안에 넣어주세요.
TOKEN = "여기에_발급받은_인증키를_붙여넣으세요"

# 2. 질병관리청 건강정보 목록 조회 API URL
url = f"http://api.kdca.go.kr/api/provide/healthInfo?TOKEN={TOKEN}"

try:
    # 3. API 서버에 데이터 요청 보내기
    response = requests.get(url)

    # 4. 수신 결과 확인하기
    if response.status_code == 200:
        # 정상적인 인증키가 아니라면 에러 메시지가 텍스트에 포함됩니다.
        if "SERVICE_KEY_IS_NOT_REGISTERED_ERROR" in response.text:
            print("⏳ 결과: 아직 시스템에 인증키가 동기화되지 않았습니다 (에러 30).")
            print("상태가 완전히 승인될 때까지 1~2시간 정도 기다린 후 다시 시도해 주세요.")
        elif "S001" in response.text or "NORMAL_CODE" in response.text:
            print("🎉 결과: API 연결 성공! 인증키가 정상적으로 작동 중입니다.")
            print("\n[수신된 데이터 미리보기]")
            print(response.text[:800]) # 데이터 앞부분 800자만 출력
        else:
            print("⚠️ 연결은 되었으나 예상치 못한 응답이 왔습니다.")
            print(response.text[:800])
    else:
        print(f"❌ 연결 실패: 서버 에러 (HTTP 상태 코드 {response.status_code})")

except Exception as e:
    print(f"❌ 실행 중 오류가 발생했습니다: {e}")

# 4단계 타겟 질환 핀셋 추출 및 RAG 전처리 코드

경추/목: 일자목(거북목)증후군 (5972)

손목: 수근굴(수근관) 증후군 (6292)

요추/허리: 요통 (3796), 추간판탈출증 (3348), 척추측만증 (3628), 척추후만증 (3629)

In [1]:
import requests
import ssl
from requests.adapters import HTTPAdapter
from google.colab import userdata
import urllib3
import warnings
import xml.etree.ElementTree as ET
import json
import time

# 1. SSL 보안 경고 숨기기 및 구형 서버 우회용 어댑터 세팅
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings('ignore')

class LegacySSLAdapter(HTTPAdapter):
    def init_poolmanager(self, *args, **kwargs):
        ctx = ssl.create_default_context()
        ctx.options |= ssl.OP_LEGACY_SERVER_CONNECT
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
        kwargs['ssl_context'] = ctx
        return super(LegacySSLAdapter, self).init_poolmanager(*args, **kwargs)

# 코랩 비밀 탭에서 키 가져오기
TOKEN = userdata.get('KDCA_API_KEY')

session = requests.Session()
session.mount('https://', LegacySSLAdapter())

# =====================================================================
# 2. 검증 완료된 VDT 핵심 질환 6개 핀셋 리스트 세팅 (척추관 협착증 제외)
# =====================================================================
target_diseases = [
    {"sn": "5972", "body_part": "경추/목", "name": "일자목(거북목)증후군"},
    {"sn": "6292", "body_part": "손목", "name": "수근굴(수근관) 증후군"},
    {"sn": "3796", "body_part": "요추/허리", "name": "요통"},
    {"sn": "3348", "body_part": "요추/허리", "name": "추간판탈출증(디스크)"},
    {"sn": "3628", "body_part": "요추/허리", "name": "척추의 형태 이상(척추측만증)"},
    {"sn": "3629", "body_part": "요추/허리", "name": "척추의 형태 이상(척추 후만증)"}
]

final_rag_dataset = []

print("🚀 기능 4단계 맞춤형 의학 지식 RAG 전처리를 시작합니다.")
print("=" * 60)

# 3. 질환별 상세정보(view) 호출 및 데이터 쪼개기(Chunking)
for idx, disease in enumerate(target_diseases, 1):
    print(f"⏳ [{idx}/{len(target_diseases)}] [{disease['body_part']}] {disease['name']} 수집 중...")

    view_url = f"https://api.kdca.go.kr/api/provide/healthInfo?TOKEN={TOKEN}&cntntsSn={disease['sn']}"

    try:
        response = session.get(view_url, verify=False)

        if response.status_code in [200, 201]:
            # XML 데이터 파싱
            root = ET.fromstring(response.text)
            chunk_count = 0

            # <cntntsCl> 태그 내부의 세부 항목(요약문, 정의, 원인, 증상 등) 순회
            for cl in root.findall('.//cntntsCl'):
                cl_nm_el = cl.find('CNTNTS_CL_NM')
                cl_cn_el = cl.find('CNTNTS_CL_CN')

                if cl_nm_el is not None and cl_cn_el is not None:
                    cl_nm = cl_nm_el.text.strip() if cl_nm_el.text else ""
                    cl_cn = cl_cn_el.text.strip() if cl_cn_el.text else ""

                    # 내용이 비어있거나 이미지 주소 데이터인 경우 필터링
                    if not cl_cn or cl_cn.startswith("http"):
                        continue

                    # 챗봇용 Q&A 쌍 및 메타데이터 구조화
                    chunk = {
                        "disease_name": disease['name'],
                        "question": f"{disease['name']}의 {cl_nm}에 대해 알려주세요.",
                        "answer": cl_cn,
                        "metadata": {
                            "body_part": disease['body_part'],  # 경추 / 요추 / 손목 필터링용
                            "category": cl_nm,                 # 정의 / 원인 / 증상 등
                            "cntnts_sn": disease['sn'],
                            "source": "질병관리청 국가건강정보포털"
                        }
                    }
                    final_rag_dataset.append(chunk)
                    chunk_count += 1

            print(f"   ➡️ 성공 (생성된 지식 청크: {chunk_count}개)")
        else:
            print(f"   ❌ 호출 실패 (HTTP 상태 코드: {response.status_code})")

    except Exception as e:
        print(f"   ❌ 에러 발생: {e}")

    # 공공데이터 서버 트래픽 방지를 위한 매너 타임(1.5초 디레이)
    time.sleep(1.5)

# =====================================================================
# 4. 고품질 JSON 파일로 영구 저장
# =====================================================================
output_file = "vdt_disease_rag_database.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(final_rag_dataset, f, ensure_ascii=False, indent=2)

print("=" * 60)
print(f"🎉 [기능 4단계] RAG 지식 베이스 구축 완벽 성공!")
print(f"📁 생성된 파일명: {output_file}")
print(f"📊 총 수집된 의미 단락(Chunk) 수: {len(final_rag_dataset)}개")
print("=" * 60)
print("💡 코랩 왼쪽 폴더 모양 아이콘을 눌러 생성된 JSON 파일을 확인하고 다운로드하세요!")

🚀 기능 4단계 맞춤형 의학 지식 RAG 전처리를 시작합니다.
⏳ [1/6] [경추/목] 일자목(거북목)증후군 수집 중...
   ➡️ 성공 (생성된 지식 청크: 13개)
⏳ [2/6] [손목] 수근굴(수근관) 증후군 수집 중...
   ➡️ 성공 (생성된 지식 청크: 20개)
⏳ [3/6] [요추/허리] 요통 수집 중...
   ➡️ 성공 (생성된 지식 청크: 14개)
⏳ [4/6] [요추/허리] 추간판탈출증(디스크) 수집 중...
   ➡️ 성공 (생성된 지식 청크: 24개)
⏳ [5/6] [요추/허리] 척추의 형태 이상(척추측만증) 수집 중...
   ➡️ 성공 (생성된 지식 청크: 6개)
⏳ [6/6] [요추/허리] 척추의 형태 이상(척추 후만증) 수집 중...
   ➡️ 성공 (생성된 지식 청크: 8개)
🎉 [기능 4단계] RAG 지식 베이스 구축 완벽 성공!
📁 생성된 파일명: vdt_disease_rag_database.json
📊 총 수집된 의미 단락(Chunk) 수: 85개
💡 코랩 왼쪽 폴더 모양 아이콘을 눌러 생성된 JSON 파일을 확인하고 다운로드하세요!
